In [10]:

import sys
sys.path.insert(0, "../src")


In [11]:
def box_to_center(bbox, stride = 4 ):
    
    x, y, w, h = bbox
    
    center_x = (x + (w/2)) / stride
    center_y = (y + (h/2)) / stride
    center_cell = (int(center_x), int(center_y))
    

    offset= (center_x - center_cell[0], center_y - center_cell[1])
    
    cell_size_x = w /stride
    cell_size_y = h / stride
    
    size = (cell_size_x, cell_size_y)
    
    
    
    return center_cell, offset, size
    

In [12]:
center_cell, offset, size = box_to_center([200, 100, 40, 20])

print(center_cell, offset, size)

(55, 27) (0.0, 0.5) (10.0, 5.0)


In [13]:
import numpy as np

heatmap = np.zeros((15,128,128), dtype= np.float32)
offset_map = np.zeros((2, 128,128), dtype= np.float32)
size_map = np.zeros((2, 128, 128), dtype= np.float32)
mask = np.zeros((128,128), dtype= np.float32)

heatmap.shape, offset_map.shape, size_map.shape, mask.shape



((15, 128, 128), (2, 128, 128), (2, 128, 128), (128, 128))

In [14]:
mask[27, 55] = 1
size_map[0, 27,55] = 10.0
size_map[1, 27, 55] = 5.0
offset_map[0, 27, 55] = 0.0
offset_map[1, 27, 55] = 0.5 

In [15]:
def draw_gaussian(heatmap_channel, cell, sigma):
    
    cx, cy = cell
    radius = int(3 * sigma) + 1
    
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            row = dy + cy
            column = dx + cx 
            
            if row >= 128 or row < 0   or column >= 128  or column < 0:
                continue
            
            value = np.exp(-(dx**2 + dy**2) / (2 * sigma**2))
            
            heatmap_channel[row, column] = max(heatmap_channel[row, column], value)
            
            
    

In [16]:
draw_gaussian(heatmap[0], (55, 27), sigma= 1  )

In [17]:
print(np.round(heatmap[0, 25:30, 53:58], 2))

[[0.02 0.08 0.14 0.08 0.02]
 [0.08 0.37 0.61 0.37 0.08]
 [0.14 0.61 1.   0.61 0.14]
 [0.08 0.37 0.61 0.37 0.08]
 [0.02 0.08 0.14 0.08 0.02]]


In [18]:
import os
import json

def build_targets(annotaitons, num_classes = 15, grid = 128, stride = 4):
    
    heatmap = np.zeros((num_classes,grid,grid), dtype= np.float32)
    offset_map = np.zeros((2, grid,grid), dtype= np.float32)
    size_map = np.zeros((2, grid, grid), dtype= np.float32)
    mask = np.zeros((grid,grid), dtype= np.float32)
    
    for annotation in annotaitons:
        bbox = annotation["bbox"] 
        cat_id = annotation["category_id"]
        
        cell, offset_size, size = box_to_center(bbox, stride)
        cx, cy = cell
        
        if cx >= grid or cy >= grid or cx <0 or cy < 0:
            continue
        
        sigma = max(min(size) / 6, 0.5)
        draw_gaussian(heatmap[cat_id - 1], cell, sigma)
        
        mask[cy, cx]= 1
        size_map[0, cy, cx] = size[0]
        size_map[1, cy, cx] = size[1]
        offset_map[0, cy,cx] = offset_size[0]
        offset_map[1, cy, cx] = offset_size[1]
        
    return heatmap, size_map, offset_map, mask
        
    
    
    
    
        
    

In [19]:
with open(os.path.expanduser("~/data/tiles/train/annotations.json")) as f:
    train_tiles = json.load(f)

tile_id = 0
tile_anns = [a for a in train_tiles["annotations"] if a["image_id"] == tile_id]

heatmap, size_map, offset_map, mask = build_targets(tile_anns)
print(len(tile_anns), mask.sum(), heatmap.max())

1 1.0 1.0


In [20]:
from collections import Counter

counts = Counter(a["image_id"] for a in train_tiles["annotations"])
tile_id, n = counts.most_common(1)[0]
print(tile_id, n)

24996 2583


In [21]:
tile_anns = [a for a in train_tiles["annotations"] if a["image_id"] == tile_id]
heatmap, size_map, offset_map, mask = build_targets(tile_anns)
print(len(tile_anns), mask.sum(), heatmap.max())

2583 2182.0 1.0
